# Guide 10: Inference and serving: running a trained model fast and cheap

*Lifecycle* · [Read the guide](https://drvgautam.github.io/primer-on-aiengineering/inference/) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/10-inference.ipynb)

This notebook holds every code block from Guide 10, in the guide's order: 10 code cells, 6 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- From training to serving — no code
- Two phases: prefill and decode — 1 code cell
- The KV cache — 4 code cells
- Why decode is limited by memory, and how batching helps — 2 code cells
- Smaller caches: sharing keys and values — 1 code cell
- Quantization: storing weights in fewer bits — 1 code cell
- Speculative decoding — 1 code cell
- The rest of the serving stack — no code

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## Two phases: prefill and decode

The code uses PyTorch 2.14.1 from the series `requirements.txt`, as of October 2026, and its blocks run in order and share variables. Run them from the repository root. The plumbing lives in `guides/inference.py`, and the blocks show the lines that carry each lesson. It runs Guide 8's toy base model, 29,275 parameters, trained on the toy world of sentences like "the cat sat on the mat". In the toy, each character is a token, as in Guides 5 to 9; real models use the subword tokens of Guide 2. The first block times the 19-character prompt "the cat sat on the " as one pass, and as 19 passes of one character each, the shape of decode.

In [2]:
import sys, math
sys.path.insert(0, "guides")                   # run from the repository root
import torch
from inference import (load_base, load_draft, encode, text, median_ms, generate_nocache, generate_cache,
                       decode_step_ms, quantize_, per_matrix, val_loss, homes_recalled, weight_kb,
                       draft_tokens, greedy_text)
torch.set_num_threads(1)                       # one CPU thread, so timings compare like with like

base = load_base()                             # Guide 8's base model, from its checkpoint
ids = encode("the cat sat on the ")            # 19 character IDs
with torch.no_grad():
    together = median_ms(lambda: base(ids[None]))                                        # one parallel pass
    one_by_one = median_ms(lambda: [base(ids[None, i:i + 1]) for i in range(len(ids))])  # 19 one-position passes
print(f"one pass {together:.2f} ms, 19 passes {one_by_one:.2f} ms, {one_by_one / together:.0f}x slower")

one pass 0.35 ms, 19 passes 5.36 ms, 15x slower


## The KV cache

The **KV cache** keeps them. Each step computes the query, key and value for the new token only, appends the key and value to a per-block store of all earlier ones, and lets the new query draw from the store. Guides 4 and 6 simplified generation as recomputing everything each step; this is the fix. Here's attention with a cache, for the toy's blocks, where `x` holds only the new positions:

In [3]:
def attend_cached(block, x, cache):            # x: B × m × d_model, the m new positions only
    B, m, d_model = x.shape
    d_k = d_model // block.h
    Q, K, V = block.qkv(x).view(B, m, 3, block.h, d_k).permute(2, 0, 3, 1, 4)   # W^Q, W^K, W^V as one matrix; each B × h × m × d_k
    if cache is not None:                      # append the new keys and values to the stored ones
        K, V = torch.cat([cache[0], K], dim=2), torch.cat([cache[1], V], dim=2)
    n = K.shape[2]                             # positions so far, cached plus new
    S = Q @ K.transpose(-2, -1) / math.sqrt(d_k)                                  # B × h × m × n
    M = torch.full((m, n), float("-inf")).triu(n - m + 1)  # m × n: new query i is at position n − m + i, so it masks later keys
    out = (torch.softmax(S + M, dim=-1) @ V).transpose(1, 2).reshape(B, m, d_model)
    return block.proj(out), (K, V)             # W^O, and the cache to keep

During prefill, `m` is the prompt's length and the mask is Guide 4's causal mask. During decode, `m` is 1 and the mask is empty: the one new query may see every stored key. `generate_cache` runs each block with this function; `generate_nocache` is Guide 6's loop. Both generate 40 characters greedily (always taking the top-scoring character, Guide 6) after "the cat sat on the ".

In [4]:
slow_text, slow_positions, slow_logits = generate_nocache(base, "the cat sat on the ", 40)
fast_text, fast_positions, fast_logits = generate_cache(base, "the cat sat on the ", 40, attend_cached)
print(repr(fast_text), fast_text == slow_text)
print(f"positions processed: {slow_positions:,} without the cache, {fast_positions} with it")
print(f"largest logit difference below 1e-4: {(slow_logits - fast_logits).abs().max() < 1e-4}")

'the cat sat on the bed and the rat lay by the log and the h' True
positions processed: 1,540 without the cache, 58 with it
largest logit difference below 1e-4: True


The same text, with about 27 times less work. Without the cache the model processes 19 + 20 + … + 58 = 1,540 positions; with it, the 19-character prompt once and then one position for each new character after the first (the prefill pass already predicts the first), 58 in all. The logits differ by less than 0.0001 (0.0000088 on our test machine), floating-point rounding from adding the numbers in a different order. The cache changes how much is computed, not what.

In [5]:
slow = median_ms(lambda: generate_nocache(base, "the cat sat on the ", 40), 10)
fast = median_ms(lambda: generate_cache(base, "the cat sat on the ", 40, attend_cached), 10)
print(f"{slow:.0f} ms without the cache, {fast:.0f} ms with it")

17 ms without the cache, 13 ms with it


The price is memory. Each token's entry holds a key and a value for every head in every block. For GPT-2 small, which has 12 blocks of 12 heads, each 64 numbers wide (Guide 5):

In [6]:
L, h, d_k = 12, 12, 64                         # GPT-2 small: blocks, heads, head width
per_token = 2 * L * h * d_k                    # a key and a value per head per block
print(f"{per_token:,} numbers per token: {per_token * 2:,} bytes in 16-bit, {per_token * 2 * 1024 / 1e6:.1f} MB at 1,024 tokens")
print(f"'The cat sat on the', 5 tokens, 32-bit: {per_token * 4 * 5:,} bytes, plus {per_token * 4:,} per new token")

18,432 numbers per token: 36,864 bytes in 16-bit, 37.7 MB at 1,024 tokens
'The cat sat on the', 5 tokens, 32-bit: 368,640 bytes, plus 73,728 per new token


## Why decode is limited by memory, and how batching helps

Recall from Guide 7 that a forward pass costs about $2N$ floating-point operations per token, for a model of $N$ parameters. **Memory bandwidth** is how many bytes per second the processor can read from memory, and a decode step has to read every weight once to produce one token. The weights, 249 MB at 16 bits even for GPT-2 small, sit in the processor's main memory, far larger than its small, fast on-chip memory, so every step streams them in again. Divide the operations by the bytes read, and compare that with the hardware's own ratio, the operations it can do per second divided by its bandwidth. A step is memory-bound when its operations per byte read fall below the hardware's:

In [7]:
N = 124_439_808                                # GPT-2 small's parameters (Guide 5)
print(f"one decode step: {2 * N / 1e6:.0f} million operations on {N * 2 / 1e6:.0f} MB of 16-bit weights, "
      f"{2 * N / (N * 2):.0f} operation per byte read")

one decode step: 249 million operations on 249 MB of 16-bit weights, 1 operation per byte read


**Batching** gives decode the same reuse. Decode many sequences together, and one read of the weights serves all of them: at a batch of $B$ sequences a step does about $B$ operations per byte, so on that 300-to-1 accelerator decode stays memory-bound until around 300 sequences. In practice the caches cap the batch first: 300 full GPT-2 contexts need about 11 GB of cache, 45 times the weights, and each sequence's cache is also read every step, which batching doesn't share. This block times one decode step at four batch sizes, each sequence with 20 tokens already cached.

In [8]:
for batch in (1, 16, 64, 256):
    ms = decode_step_ms(base, batch, attend_cached)                  # median of 30 steps
    print(f"batch {batch:3}: {ms:.2f} ms per step, {batch / ms * 1000:,.0f} tokens per second")

batch   1: 0.31 ms per step, 3,239 tokens per second
batch  16: 0.42 ms per step, 37,934 tokens per second
batch  64: 0.71 ms per step, 89,530 tokens per second
batch 256: 1.98 ms per step, 129,605 tokens per second


## Smaller caches: sharing keys and values

Take a configuration like the 70-billion-parameter Llama 2 model, as published in 2023: 80 blocks (papers say layers), 64 query heads of width 128, and 8 key–value heads ([Touvron et al., 2023](https://arxiv.org/abs/2307.09288)).

In [9]:
for h_kv in (64, 8):                           # 80 blocks, 64 query heads of width 128; 64 = no sharing, for comparison
    per_token = 2 * 80 * h_kv * 128 * 2        # key and value, 16-bit: bytes per token
    print(f"h_kv = {h_kv:2}: {per_token:,} bytes per token, {per_token * 4096 / 1e9:.2f} GB at 4,096 tokens")
print(f"weights 140 GB + 16 sequences of 4,096 tokens: {140 + 16 * per_token * 4096 / 1e9:.1f} GB in all")

h_kv = 64: 2,621,440 bytes per token, 10.74 GB at 4,096 tokens
h_kv =  8: 327,680 bytes per token, 1.34 GB at 4,096 tokens
weights 140 GB + 16 sequences of 4,096 tokens: 161.5 GB in all


## Quantization: storing weights in fewer bits

This is the simplest scheme, symmetric rounding with one scale per row; the accordion after the results covers what real methods add. Each run reports validation loss on held-out world text and "homes": the model completes "the *animal* lives in the" for all six animals, as Guide 8 did.

In [10]:
def quantize_rows(W, bits):                    # W: d_out × d_in (PyTorch's nn.Linear layout)
    scale = W.abs().amax(dim=1, keepdim=True) / (2 ** (bits - 1) - 1)   # one scale per row: its largest value maps to the top level
    return (W / scale).round() * scale         # round to an integer level, then multiply back

print(f"32 bits: loss {val_loss(base):.3f}; homes {homes_recalled(base)} of 6; {weight_kb(base, 0, 32):.1f} KB")
for bits in (8, 6, 4, 3, 2):
    rows, whole = load_base(), load_base()
    count = quantize_(rows, quantize_rows, bits)                     # every weight matrix and embedding table
    quantize_(whole, per_matrix(quantize_rows), bits)                # the same, one scale per matrix
    print(f"{bits} bits: loss {val_loss(rows):.3f} per row, {val_loss(whole):.3f} per matrix; "
          f"homes {homes_recalled(rows)} of 6; {weight_kb(rows, count, bits):.1f} KB")

32 bits: loss 0.244; homes 6 of 6; 117.1 KB


8 bits: loss 0.244 per row, 0.244 per matrix; homes 6 of 6; 34.8 KB


6 bits: loss 0.244 per row, 0.245 per matrix; homes 6 of 6; 27.7 KB


4 bits: loss 0.259 per row, 0.456 per matrix; homes 6 of 6; 20.6 KB


3 bits: loss 0.937 per row, 1.500 per matrix; homes 4 of 6; 17.1 KB


2 bits: loss 4.765 per row, 6.220 per matrix; homes 0 of 6; 13.6 KB


## Speculative decoding

The draft here is a much smaller toy model, 5,227 parameters (one block, width 16), trained on the same world and loaded from a checkpoint. It proposes 4 characters at a time:

In [11]:
draft = load_draft()                           # 5,227 parameters, one block of width 16
print(f"draft loss {val_loss(draft):.3f}, large model {val_loss(base):.3f}")
def speculative(large, draft, prompt, n_new, k=4):
    ids, passes, accepted = encode(prompt), 0, 0
    while len(ids) < len(prompt) + n_new:
        guess = draft_tokens(draft, ids, k)                          # the draft's next k characters
        with torch.no_grad():                                        # one large-model pass checks all k
            choice = large(torch.cat([ids, guess])[None])[0, len(ids) - 1:].argmax(-1)   # k + 1 choices
        ok = 0
        while ok < k and guess[ok] == choice[ok]:                    # longest run the large model agrees with
            ok += 1
        ids = torch.cat([ids, guess[:ok], choice[ok:ok + 1]])        # kept guesses, then the large model's own choice
        passes, accepted = passes + 1, accepted + ok
    return text(ids[:len(prompt) + n_new]), passes, accepted

for prompt in ("the cat sat on the ", "the dog ran by the ", "the owl lives in the ", "the fox hid under the "):
    out, passes, accepted = speculative(base, draft, prompt, 38)
    same = out == prompt + greedy_text(base, prompt, 38)
    print(f"{out!r} same as greedy: {same}; {passes} passes, {accepted} of {4 * passes} guesses kept")

draft loss 0.269, large model 0.244
'the cat sat on the bed and the rat lay by the log and the' same as greedy: True; 10 passes, 28 of 40 guesses kept
'the dog ran by the log and the rat lives in the log and t' same as greedy: True; 11 passes, 29 of 44 guesses kept


'the owl lives in the pot and the hen sat by the mat and the' same as greedy: True; 10 passes, 31 of 40 guesses kept
'the fox hid under the mat and the rat lives in the log and t' same as greedy: True; 10 passes, 30 of 40 guesses kept


---

← [Guide 9: Alignment](09-alignment.ipynb) · [Guide 11: Tool use and agents](11-tools.ipynb) →

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).